# Backbone and Training-Regime Comparison

Compare backbone choices (ResNet18/50), frozen vs fine-tuned training, and
embedding dimensions (256 vs 512) on LeaData V1.

> **Warning:** Results are diagnostic only. LeaData is a public leather texture
> dataset, not a luxury authentication dataset.

## Run Experiments

From `auth-platform/`:

```bash
# Quick (2 epochs, 100 images, 8 configs)
make backbone-comparison-leadata-quick

# Medium (5 epochs, 300 images)
make backbone-comparison-leadata-medium

# Full
make backbone-comparison-leadata

# Compare against baseline
make compare-backbone-comparison
```

In [ ]:
import json
from pathlib import Path

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

BB_DIR = Path("../experiments/outputs/backbone-comparison")
summary_path = BB_DIR / "backbone_comparison_summary.json"
print(f"Exists: {summary_path.exists()}")

In [ ]:
if not summary_path.exists():
    print("No results found. Run: make backbone-comparison-leadata-quick")
    data = None
    results = []
else:
    data = json.loads(summary_path.read_text())
    results = [r for r in data.get("results", []) if r.get("evaluation_metrics")]
    print(f"Results: {len(results)}  Failures: {len(data.get('failures', []))}")
    print()

    header = f"{'Backbone':<10} {'Freeze':>6} {'Dim':>5} {'Loss':<22} {'Trainable':>12} {'ClustSep':>10} {'NN Agree':>10} {'R@1':>8}"
    print(header)
    print("-" * len(header))
    for r in sorted(results, key=lambda x: x["evaluation_metrics"].get("cluster_separation", 0), reverse=True):
        em = r["evaluation_metrics"]
        print(f"{r.get('backbone', '?'):<10} "
              f"{'yes' if r.get('freeze_backbone') else 'no':>6} "
              f"{r.get('embedding_dim', ''):>5} "
              f"{r.get('loss', '?'):<22} "
              f"{r.get('trainable_params', '?'):>12} "
              f"{em.get('cluster_separation', 0):>10.4f} "
              f"{em.get('nn_agreement_rate', 0):>9.1%} "
              f"{em.get('recall_at_1', 0):>8.4f}")

In [ ]:
# Cluster separation by config
if results:
    labels_plot = [f"{r.get('backbone','?')} {'frozen' if r.get('freeze_backbone') else 'finetune'}\n{r.get('loss','?')} {r.get('embedding_dim','')}d"
                   for r in sorted(results, key=lambda x: x["evaluation_metrics"].get("cluster_separation", 0))]
    seps = [r["evaluation_metrics"].get("cluster_separation", 0)
            for r in sorted(results, key=lambda x: x["evaluation_metrics"].get("cluster_separation", 0))]

    fig, ax = plt.subplots(figsize=(10, max(4, len(results) * 0.7)))
    bars = ax.barh(labels_plot, seps, color="steelblue")
    ax.axvline(x=0.9418, color="red", linestyle="--", label="texture_baseline_v0.2.0 (0.9418)")
    ax.set_xlabel("Cluster Separation")
    ax.set_title("Backbone Comparison — Cluster Separation")
    ax.legend()
    for bar, val in zip(bars, seps):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

## Notes / Conclusions

- Compares ResNet18 vs ResNet50, frozen vs fine-tuned, 256d vs 512d.
- Contrastive and batch-hard triplet losses both tested.
- Frozen backbone trains projection head only — faster, fewer parameters.
- `texture_baseline_v0.2.0` (0.9418) is the reference line on all charts.

**These results are diagnostic only.** They do not prove authentication accuracy.